In [1]:
# pandas 등 데이터 분석에 필요한 도구(라이브러리)들을 불러옴
# 우리 프로젝트 폴더로 작업 위치(경로)를 옮김
# data 폴더 안에 있는 원본 파일 목록(경찰청, e지방지표)을 확인함
import pandas as pd   # 표(DataFrame) 형태로 데이터를 다루는 라이브러리
import glob           # 폴더 안의 파일 목록을 패턴으로 찾는 라이브러리
import os             # 폴더 경로, 작업 위치 등을 다루는 라이브러리
import io             # 파일을 메모리에서 텍스트처럼 다루기 위한 라이브러리


os.chdir('/Users/ijaejun/Documents/sophomore_high/crime_catchers')
# os.chdir(경로): 현재 작업 폴더를 지정한 경로로 바꿔주는 함수
#   └ 이후 모든 상대경로(예: 'data/...')는 이 폴더를 기준으로 찾게 됨
print("현재 위치:", os.getcwd())
# os.getcwd(): 지금 작업 중인 폴더의 경로를 알려주는 함수
print("파일 목록:", glob.glob('data/raw/경찰청/*.csv'))
# glob.glob(패턴): 패턴에 맞는 파일 경로를 모두 찾아 리스트로 반환하는 함수
#   └ '*.csv' : 확장자가 csv인 모든 파일을 의미 (경찰청 범죄 데이터 파일 목록 확인)
print(os.getcwd())
print(glob.glob('data/raw/e지방지표/*.csv'))
# 위와 동일한 함수로, e지방지표(독립변수) 폴더의 csv 파일 목록을 확인

현재 위치: /Users/ijaejun/Documents/sophomore_high/crime_catchers
파일 목록: ['data/raw/경찰청/경찰청_범죄_발생_2016.csv', 'data/raw/경찰청/경찰청_범죄_발생_2017.csv', 'data/raw/경찰청/경찰청_범죄_발생_2015.csv', 'data/raw/경찰청/경찰청_범죄_발생_2014.csv', 'data/raw/경찰청/경찰청_범죄_발생_2023.csv', 'data/raw/경찰청/경찰청_범죄_발생_2022.csv', 'data/raw/경찰청/경찰청_범죄_발생_2020.csv', 'data/raw/경찰청/경찰청_범죄_발생_2021.csv', 'data/raw/경찰청/경찰청_범죄_발생_2019.csv', 'data/raw/경찰청/경찰청_범죄_발생_2024.csv', 'data/raw/경찰청/경찰청_범죄_발생_2018.csv']
/Users/ijaejun/Documents/sophomore_high/crime_catchers
['data/raw/e지방지표/대구_e지방지표.csv', 'data/raw/e지방지표/부산_e지방지표.csv', 'data/raw/e지방지표/인천_e지방지표.csv', 'data/raw/e지방지표/대전_e지방지표.csv', 'data/raw/e지방지표/광주_e지방지표.csv', 'data/raw/e지방지표/울산_e지방지표.csv']


In [2]:
# 6개 도시의 e지방지표 파일을 하나씩 읽어서
# 우리가 분석에 쓸 변수(실업률, 음주율 등)만 골라내고
# 연도별로 정리해서 하나의 표(지역+연도가 행이 되는 표)로 합침
# ============================================================
# Step 1. e지방지표 - 독립변수 합치기
# ============================================================

# INDICATORS: 원본 파일의 긴 지표 이름을 분석에서 쓸 짧은 변수 이름으로 바꿔주는 사전(딕셔너리)
#   └ key(왼쪽)   : e지방지표 원본 파일에 적힌 지표 이름
#   └ value(오른쪽) : 분석에서 사용할 변수 이름
INDICATORS = {
    '실업률(시도) (%)':           '실업률',
    '음주율 (%)':                 '음주율',
    '소비자물가 등락률 (%)':       '물가상승률',
    '주민등록인구 (명)':           '인구수',
    '기온 (℃)':                  '평균기온',
    '경찰공무원 1인당 담당주민수 (명)': '경찰1인당주민수',
    '국민기초생활보장 수급자수 (명)': '기초수급자수',
    '조이혼율':                   '조이혼율',
    '1인당 GRDP (천원)':          '지역소득',
    '등록외국인 현황 (명)':        '외국인수',
}

YEARS = [str(y) for y in range(2014, 2025)]  # 2014~2024
# [str(y) for y in range(2014, 2025)] : 2014~2024년을 문자열 리스트로 만듦
#   └ 파일의 연도 컬럼 이름이 문자열('2014' 등)이라서 같은 형식으로 맞춰줌
e_files = glob.glob('data/raw/e지방지표/*.csv')
# glob.glob(): 'data/raw/e지방지표/' 폴더 안의 모든 csv 파일 경로를 리스트로 가져옴
e_all = []

for file in e_files:
    city = os.path.basename(file).replace('_e지방지표.csv', '')
    # os.path.basename(): 전체 경로에서 파일 이름만 뽑아냄 (예: '광주_e지방지표.csv')
    #   └ .replace('_e지방지표.csv', '') : 뒤에 붙은 '_e지방지표.csv' 글자를 지워서 도시 이름만 남김
    df = pd.read_csv(file, encoding='utf-8-sig')
    # read_csv(): CSV 파일을 표(DataFrame)로 읽어오는 함수
    #   └ encoding='utf-8-sig' : 한글이 깨지지 않도록 글자 인코딩을 지정

    # 필요한 지표만 필터링
    df_filtered = df[df['지표별'].isin(INDICATORS.keys())].copy()
    # isin(INDICATORS.keys()) : '지표별' 컬럼의 값이 INDICATORS 사전의 key(우리가 필요한 지표) 중 하나면 True
    #   └ df[조건] : True인 행만 골라냄
    #   └ .copy()  : 원본 표와 분리된 새 표를 만들어 안전하게 수정

    # 연도 컬럼만 추출
    year_cols = [y for y in YEARS if y in df_filtered.columns]
    # 리스트 컴프리헨션 : YEARS 중에서 실제로 df_filtered에 존재하는 연도 컬럼만 골라냄
    df_filtered = df_filtered[['지표별'] + year_cols]
    # ['지표별'] + year_cols : '지표별' 컬럼과 연도 컬럼들만 남기고 나머지는 버림

    # 세로로 변환 (wide → long)
    df_long = df_filtered.melt(
        id_vars='지표별',
        value_vars=year_cols,
        var_name='연도',
        value_name='값'
    )
    # melt(): 옆으로 넓게 퍼진 표(연도별로 컬럼이 나뉜 표)를 세로로 긴 표로 바꿔주는 함수
    #   └ id_vars='지표별'      : 그대로 유지할 기준 컬럼
    #   └ value_vars=year_cols  : 세로로 펼칠 대상이 되는 연도 컬럼들
    #   └ var_name='연도'       : 펼쳐진 컬럼 이름들이 모일 새 컬럼 이름
    #   └ value_name='값'       : 각 셀의 값들이 모일 새 컬럼 이름

    df_long['지역'] = city
    # 새 컬럼 '지역'에 현재 파일의 도시 이름을 채워 넣음
    df_long['연도'] = df_long['연도'].astype(int)
    # astype(int) : 문자열로 되어 있던 연도를 숫자(정수)로 바꿔줌
    df_long['지표별'] = df_long['지표별'].map(INDICATORS)
    # map(사전) : '지표별' 컬럼의 각 값을 INDICATORS 사전을 이용해 짧은 변수 이름으로 바꿔줌
    e_all.append(df_long)
    # append() : 완성된 도시별 표를 리스트(e_all)에 추가

    # 위 코드 그대로 + 아래 추가!

df_e = pd.concat(e_all, ignore_index=True)
# concat(): 여러 개의 표(리스트 안의 DataFrame들)를 위아래로 이어붙여 하나의 표로 만듦
#   └ ignore_index=True : 이어붙인 후 행 번호(인덱스)를 0부터 다시 매김

df_e_pivot = df_e.pivot_table(
    index=['지역', '연도'],
    columns='지표별',
    values='값',
    aggfunc='first'
).reset_index()
# pivot_table(): 세로로 긴 표를 다시 넓은 표로 바꿔주는 함수 (긴 표 → 넓은 표)
#   └ index=['지역','연도'] : 표의 행(세로) 기준이 되는 컬럼
#   └ columns='지표별'      : 표의 새 열(가로) 이름이 될 컬럼
#   └ values='값'           : 표 안에 채워질 실제 값
#   └ aggfunc='first'       : 같은 지역+연도+지표가 여러 개면 첫 번째 값만 사용
#   └ .reset_index()        : '지역','연도'를 인덱스에서 다시 일반 컬럼으로 되돌림

df_e_pivot.columns.name = None
# columns.name = None : pivot_table을 거치며 생긴 컬럼 그룹 이름('지표별')을 없애서 표를 깔끔하게 정리

print("✅ e지방지표 완료!")
print(df_e_pivot.shape)
# shape : 표의 (행 개수, 열 개수)를 알려주는 속성
print(df_e_pivot.head())
# head() : 표의 맨 위 5개 행을 미리 보여주는 함수

✅ e지방지표 완료!
(66, 12)
   지역    연도 경찰1인당주민수 기초수급자수 물가상승률  실업률   외국인수   음주율      인구수 조이혼율   지역소득  평균기온
0  광주  2014    476.7  59598   1.6  2.8  17064  60.6  1475884  2.1  23448  14.3
1  광주  2015    458.6  71683   0.3  2.9  18455  61.9  1472199  1.9  24731  14.6
2  광주  2016    454.6  69420   0.9  3.1  19920  58.6  1469214  1.9  26248  15.0
3  광주  2017    447.5  65712   2.1  2.9  21279  61.6  1463770  1.8  27449  14.6
4  광주  2018    439.6  72757   1.2  3.8  22815  60.3  1459336  2.0  28594  14.6


In [3]:
# 경찰청 범죄 데이터에서 5대 범죄(살인기수, 강도, 강간, 절도, 폭행)의 건수를
# 도시별·연도별로 더해서 하나의 표로 정리함
# 절도는 연도에 따라 컬럼 안의 이름이 '절도' 또는 '절도범죄'로 달라서, 실제로 있는 이름을 자동으로 찾아서 사용함
CRIMES_5 = ['살인기수', '강도', '강간', '절도', '폭행']
CITIES = ['부산', '대구', '인천', '광주', '대전', '울산']

crime_files = glob.glob('data/raw/경찰청/*.csv')
crime_all = []

for file in sorted(crime_files):
    # sorted(): 파일 목록을 이름 순서(=연도 순서)대로 정렬해서 하나씩 처리
    year = int(''.join(filter(str.isdigit, os.path.basename(file)))[:4])
    # os.path.basename() : 파일 경로에서 파일 이름만 추출 (예: '경찰청_범죄_발생_2014.csv')
    #   └ filter(str.isdigit, ...) : 파일 이름 글자 중 숫자만 골라냄
    #   └ ''.join(...) : 골라낸 숫자들을 하나의 문자열로 합침
    #   └ [:4]  : 앞의 4글자(연도)만 사용
    #   └ int() : 문자열을 숫자(정수)로 바꿔서 연도 값으로 사용

    df = None
    for enc in ['cp949', 'utf-8-sig', 'utf-8']:
        try:
            temp = pd.read_csv(file, encoding=enc)
            # read_csv(): CSV 파일을 표(DataFrame)로 읽어오는 함수
            #   └ encoding=enc : 파일마다 글자 인코딩이 다를 수 있어 여러 방식을 차례로 시도
            if '범죄중분류' in temp.columns:
                df = temp
                break
        except:
            continue

    if df is None:
        print(f"❌ 실패: {file}")
        continue

    # 2014~2017 파일은 '절도', 2018~2024 파일은 '절도범죄'로 표기되므로
    # 연도별로 실제 존재하는 이름을 자동으로 찾는다.
    theft_col = '절도' if '절도' in df['범죄중분류'].values else '절도범죄'
    # df['범죄중분류'].values : '범죄중분류' 컬럼의 모든 값을 배열로 가져옴
    #   └ '절도' in (배열) : 그 배열 안에 '절도'라는 값이 들어있는지 확인
    #   └ 있으면 theft_col = '절도', 없으면(2018~2024) theft_col = '절도범죄'

    for city in CITIES:
        city_cols = [c for c in df.columns if c.startswith(city)]
        # c.startswith(city) : 컬럼 이름이 해당 도시 이름으로 시작하는지 확인 (그 도시의 구·군 컬럼들을 모음)
        row = {'지역': city, '연도': year}

        # 범죄별 개별 집계
        for crime in CRIMES_5:
            crime_name = theft_col if crime == '절도' else crime
            # crime이 '절도'면 위에서 찾은 theft_col('절도' 또는 '절도범죄')을 사용, 아니면 그대로 사용
            df_crime_row = df[df['범죄중분류'] == crime_name]
            # df[조건] : '범죄중분류' 컬럼 값이 crime_name과 같은 행만 골라냄
            if not df_crime_row.empty:
                row[crime] = int(df_crime_row[city_cols].sum().sum())
                # sum().sum() : 해당 도시의 모든 구·군 컬럼 값을 먼저 더하고(행 방향), 그 결과를 다시 더함(전체 합)
                #   └ int() : 결과를 정수로 변환
            else:
                row[crime] = 0

        crime_all.append(row)

df_crime = pd.DataFrame(crime_all).sort_values(['연도', '지역']).reset_index(drop=True)
# pd.DataFrame(): 리스트(crime_all) 안의 딕셔너리들을 표(DataFrame)로 만듦
#   └ .sort_values(['연도','지역']) : 연도, 지역 순서로 행을 정렬
#   └ .reset_index(drop=True)       : 정렬 후 행 번호(인덱스)를 0부터 다시 매기고, 기존 번호는 버림

zero_theft = df_crime[df_crime['절도'] == 0][['지역', '연도', '절도']]
# 절도 건수가 0인 행만 골라냄 (이름 매칭이 잘못되면 0이 나올 수 있어 점검용)
if not zero_theft.empty:
    raise ValueError(f"절도 집계가 0인 행이 있습니다. 원본 범죄명 매칭을 확인하세요:\n{zero_theft}")
    # raise ValueError(): 조건이 맞으면 에러를 발생시켜 코드를 멈추고 문제를 알려줌
print("✅ 완료!")
print(df_crime.head(6))
# head(6) : 표의 맨 위 6개 행만 보여주는 함수 (6개 도시의 2014년 데이터 확인용)

✅ 완료!


   지역    연도  살인기수   강도   강간     절도    폭행
0  광주  2014    12   53  220  10361  5718
1  대구  2014    17   63  219  14609  5280
2  대전  2014    12   82  168  11453  2457
3  부산  2014    21  131  399  21898  7821
4  울산  2014    13   32  126   5777  3654
5  인천  2014    16  111  308   9737  8300


In [4]:
# e지방지표 표(독립변수)와 범죄 데이터 표(종속변수)를 지역+연도 기준으로 하나로 합침(merge)
# 5대 범죄 건수를 모두 더해서 '5대범죄합계' 컬럼을 새로 만듦
# 외국인수를 인구수로 나눠서 외국인비율(%)을 계산하고, 컬럼 순서를 정리함
# ============================================================
# Step 3. 두 데이터 합치기 (merge)
# ============================================================

df_merged = pd.merge(df_e_pivot, df_crime, on=['지역', '연도'], how='inner')
# pd.merge(): 두 표를 공통된 기준 컬럼으로 합치는 함수
#   └ on=['지역','연도'] : '지역'과 '연도' 값이 같은 행끼리 연결
#   └ how='inner'        : 두 표 모두에 존재하는 지역+연도 조합만 남김
df_merged['5대범죄합계'] = (
    df_merged['살인기수']
    + df_merged['강도']
    + df_merged['강간']
    + df_merged['절도']
    + df_merged['폭행']
)
# 5대 범죄(살인기수, 강도, 강간, 절도, 폭행) 건수를 모두 더해서 '5대범죄합계' 컬럼을 새로 만듦
# 숫자 변환
num_cols = ['외국인수', '인구수', '실업률', '음주율', '물가상승률',
            '평균기온', '경찰1인당주민수', '기초수급자수', '조이혼율', '지역소득']
for col in num_cols:
    df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce')
    # pd.to_numeric(): 문자열로 되어 있을 수 있는 값을 진짜 숫자로 바꿔주는 함수
    #   └ errors='coerce' : 숫자로 바꿀 수 없는 값은 NaN(결측치)으로 처리

# 외국인 비율
df_merged['외국인비율(%)'] = (
    df_merged['외국인수'] / df_merged['인구수'] * 100
).round(4)
# (외국인수 ÷ 인구수) × 100 : 전체 인구 중 외국인 비율(%)을 계산
#   └ round(4) : 계산 결과를 소수점 4자리까지만 남김

print("✅ 최종 완료!")
print(df_merged.shape)
# shape : 표의 (행 개수, 열 개수)를 알려주는 속성
print(df_merged.head())
# head() : 표의 맨 위 5개 행을 미리 보여주는 함수

# 컬럼 순서 정리
cols_order = [
    '지역', '연도',
    '5대범죄합계',        # 종속변수
    '실업률', '음주율', '물가상승률', '인구수',
    '평균기온', '경찰1인당주민수', '기초수급자수',
    '조이혼율', '지역소득', '외국인수', '외국인비율(%)'
]
df_merged = df_merged[cols_order]
# df_merged[cols_order] : 컬럼 순서를 cols_order 리스트 순서대로 다시 정렬함

print("\n✅ 최종 병합 완료!")
print(df_merged.shape)
print(df_merged)


✅ 최종 완료!
(66, 19)
   지역    연도  경찰1인당주민수  기초수급자수  물가상승률  실업률   외국인수   음주율      인구수  조이혼율   지역소득  \
0  광주  2014     476.7   59598    1.6  2.8  17064  60.6  1475884   2.1  23448   
1  광주  2015     458.6   71683    0.3  2.9  18455  61.9  1472199   1.9  24731   
2  광주  2016     454.6   69420    0.9  3.1  19920  58.6  1469214   1.9  26248   
3  광주  2017     447.5   65712    2.1  2.9  21279  61.6  1463770   1.8  27449   
4  광주  2018     439.6   72757    1.2  3.8  22815  60.3  1459336   2.0  28594   

   평균기온  살인기수  강도   강간     절도    폭행  5대범죄합계  외국인비율(%)  
0  14.3    12  53  220  10361  5718   16364    1.1562  
1  14.6    11  44  216   8438  5386   14095    1.2536  
2  15.0     9  47  170   6050  4797   11073    1.3558  
3  14.6     3  33  203   4821  4856    9916    1.4537  
4  14.6    10  27  197   4832  5004   10070    1.5634  

✅ 최종 병합 완료!
(66, 14)
    지역    연도  5대범죄합계  실업률   음주율  물가상승률      인구수  평균기온  경찰1인당주민수  기초수급자수  조이혼율  \
0   광주  2014   16364  2.8  60.6    1.6  1475884  14.3     476.

In [5]:
# 지금까지 합친 전체 데이터를 data/processed 폴더에 CSV 파일로 저장함
# 이 파일(merged_data.csv)은 다음 단계인 02_preprocessing.ipynb에서 사용됨
# ============================================================
# Step 4. 저장
# ============================================================

os.makedirs('data/processed', exist_ok=True)
# os.makedirs(): 폴더를 새로 만드는 함수
#   └ exist_ok=True : 폴더가 이미 있어도 에러를 내지 않고 그냥 넘어감
df_merged.to_csv('data/processed/merged_data.csv',
                 index=False, encoding='utf-8-sig')
# to_csv(): 표(DataFrame)를 CSV 파일로 저장하는 함수
#   └ index=False          : 행 번호(인덱스)는 파일에 저장하지 않음
#   └ encoding='utf-8-sig' : 한글이 깨지지 않게 저장
print("\n💾 data/processed/merged_data.csv 저장 완료!")


💾 data/processed/merged_data.csv 저장 완료!
